In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler


In [2]:
# Load cleaned merged data
df = pd.read_csv("merged_cleaned_data.csv")


Create New Features

In [3]:
# Property Age
if 'year_built' in df.columns:
    df['property_age'] = 2025 - df['year_built']

In [4]:
# Price per square foot
if 'area_sqft' in df.columns and 'final_price' in df.columns:
    df['price_per_sqft'] = df['final_price'] / df['area_sqft']


In [5]:
# Broker performance score
if 'experience_years' in df.columns and 'rating' in df.columns:
    df['broker_score'] = df['experience_years'] * df['rating']

In [6]:
# Customer income category
if 'annual_income' in df.columns:
    df['income_category'] = pd.cut(df['annual_income'],
                                   bins=[0, 100000, 200000, 400000, 1000000],
                                   labels=['Low', 'Mid', 'High', 'Luxury'])

In [7]:
# Loan rate to price ratio (if applicable)
if 'loan_rate' in df.columns and 'final_price' in df.columns:
    df['loan_to_price'] = df['loan_rate'] / df['final_price']

Handle Categorical Encoding

In [8]:
categorical_cols = ['city', 'property_type', 'furnishing', 'status',
                    'condition', 'segment', 'income_category', 'mortgage', 'channel']

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.title()

In [9]:
# Use Label Encoding for simple string categories
label_enc = LabelEncoder()
for col in categorical_cols:
    if col in df.columns:
        df[col] = label_enc.fit_transform(df[col])

Drop Useless or Text-Heavy Columns

In [10]:
drop_cols = ['broker_name', 'agency', 'full_name', 'email', 'phone',
             'notes_freeform', 'lead_source']
df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True)


Handle Missing Values Again (After Transformations)

In [11]:
df.fillna(df.median(numeric_only=True), inplace=True)


Feature Scaling

In [12]:
num_cols = df.select_dtypes(include=[np.number]).columns
scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

Split into Train/Test

In [13]:
target = 'final_price'
X = df.drop(columns=[target])
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("✅ Feature engineering complete!")
print("Training data shape:", X_train.shape)
print("Test data shape:", X_test.shape)

✅ Feature engineering complete!
Training data shape: (32000, 43)
Test data shape: (8000, 43)


In [14]:
# Save both train and test sets
train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)

train.to_csv("train_ready.csv", index=False)
test.to_csv("test_ready.csv", index=False)

print("💾 Saved 'train_ready.csv' and 'test_ready.csv' for model training & evaluation.")


💾 Saved 'train_ready.csv' and 'test_ready.csv' for model training & evaluation.
